# On-policy distillation

**Notebook 04 · Learn from the responses the student actually writes**

The student samples an answer. A frozen teacher scores possible next tokens after
those same sampled prefixes. We update only the student. This recreates GKD's
central feedback loop with a small, controlled Qwen experiment.

| Fixed decision | Choice |
| :--- | :--- |
| Models | Qwen3.5-0.8B-Base student and Qwen3.8-27B teacher |
| Starting point | Verified fresh 03 checkpoint for the selected profile |
| Teacher targets / student sampling | Temperature 0.1 / temperature 1 |
| Objective | Forward KL with explicit EOS grouping |
| Comparison | Supervised KD on references versus on-policy KD on student responses |
| Pilots | Learning rates `1e-6` and `5e-6`, up to 32 updates each |
| Full horizon | 128 updates, eight responses per update |
| Held-out evaluation | After both fixed full arms complete |

![On-policy feedback loop](assets/04_on_policy_distillation/on_policy_workflow.png)

*The response comes from the current student. The teacher supplies distributions
on its prefixes. Sampling receives no gradient.*

The [historical notebook](https://github.com/igoldin670/on-policy-distillation/blob/34f5aa3af61f840c5edca550cfd4ff2ca2521ad7/notebooks/04_on_policy_distillation.ipynb)
and [audited EOS comparison](assets/04_on_policy_distillation/recorded_eos_comparison/README.md)
retain the original models' results. New-model GPU results remain pending.

## 1. Connect to the allocated runtime

Use the same user environment as 05 and 03. Inspect mode loads data and tokenizers.
Training requires the allocated CUDA kernel. Model cache belongs on scratch;
checkpoints and reports belong in the persistent project results directory.

In [ ]:
import os
import sys
from pathlib import Path

# A remote Jupyter kernel uses files on the remote machine. Find that checkout
# before importing helpers. Uploading a project ZIP is optional.
PROJECT_ROOT = next(
    (
        p
        for p in (
            Path.cwd(),
            *Path.cwd().parents,
            Path.home() / "on-policy-distillation",
            Path("/content/on-policy-distillation"),
        )
        if (p / "src/pipeline.py").is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, "Sync or clone the full project before running this notebook."
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# Set caches before importing Hugging Face. Amarel scratch is used only for cache.
scratch_directory = Path("/scratch") / os.environ.get("USER", "")
IN_AMAREL = bool(
    os.environ.get("SLURM_JOB_ID") and os.environ.get("USER") and scratch_directory.is_dir()
)
cache_directory = (
    scratch_directory / "huggingface" if IN_AMAREL else PROJECT_ROOT / ".cache/source-verification"
)
os.environ.setdefault("HF_HOME", str(cache_directory))
os.environ.setdefault("HF_HUB_CACHE", str(Path(os.environ["HF_HOME"]) / "hub"))
os.environ.setdefault("HF_DATASETS_CACHE", str(Path(os.environ["HF_HOME"]) / "datasets"))
cache_directory.mkdir(parents=True, exist_ok=True)
# Public pinned downloads do not need Colab's secret-vault integration.
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache/matplotlib"))
MODEL_PROFILE = os.environ.get("OPD_MODEL_PROFILE", "qwen38")
print("Project", PROJECT_ROOT)
print("Profile", MODEL_PROFILE)
print("Cache", os.environ["HF_HOME"])

MODE = os.environ.get("OPD_TRAIN_MODE", "inspect")
assert MODE in {"inspect", "train"}
SFT_RUN_DIRECTORY = None  # Optional explicit completed 03 run folder.
RESUME_RUN_DIRECTORY = None  # Set the previous 04 run folder when resuming.
RUN_STAGE = "pilot"  # Switch to "full" after a qualifying pilot.

## 2. Freeze the questions and the experiment

The new 1,024 practice questions exclude SFT practice, validation, and official test
questions. The same 64 validation questions guide pilot review. A separate 256-question
test excludes the historically inspected subset. Fixed seeds, pinned revisions,
source fingerprints, and checksum inventories record exactly what was run.

Reference solutions must fit the response budget completely. Overlong examples
are recorded as exclusions. A shorter accumulation group still contributes its
actual number of responses.

In [ ]:
from dataclasses import asdict

import pandas as pd
import torch
from IPython.display import display

from src.pipeline import prepare_cold_start, prepare_distillation, run_distillation
from src.training_safety import TrainingSafetyConfig

# Data preparation is a CPU operation. It uses no student or teacher weights.
state = prepare_distillation(prepare_cold_start(PROJECT_ROOT, MODEL_PROFILE))
print("Models", state["profile"]["student_model_id"], state["profile"]["teacher_model_id"])
print(
    {
        name: len(state["splits"][name])
        for name in ("train", "validation", "primary_test", "historical_test")
    }
)
print("Safety policy", asdict(TrainingSafetyConfig()))

## 3. Match the teacher's next-token distribution

For each question $x$, the student samples a response $y$. At response position
$t$, both models read the question and the same preceding response IDs $y_{<t}$.
The teacher provides $q_t$ and the trainable student provides $p_{\theta,t}$.

$$
L_{\mathrm{GKD}}=\frac{1}{B}\sum_{b=1}^{B}\frac{1}{R_b}\sum_{t=1}^{R_b}
D_{\mathrm{KL}}(q_{b,t}\Vert p_{\theta,b,t}).
$$

For raw-token forward KL, each vocabulary contribution is
$q(v)[\log q(v)-\log p(v)]$. We average positions within each response, then average
responses. This prevents longer responses from receiving greater total weight.
Teacher probabilities are normalized after dividing teacher logits by 0.1.
Student probabilities use temperature 1. The implementation adds no temperature-squared factor.

The paper's on-policy setting uses student-generated responses and does not
backpropagate through sampling. Our supervised-KD control instead reads reference
responses with the same teacher, objective, questions, and optimizer recipe.
This isolates the source of response prefixes. [GKD Algorithm 1 and Appendix A.4](https://arxiv.org/html/2306.13649v3)

### A numerical example

If the teacher probabilities are `[0.7, 0.2, 0.1]` and student probabilities are
`[0.5, 0.3, 0.2]`, the loss compares all three possible next tokens. It remains a
full-distribution comparison even when only one token appeared in the response.

In [ ]:
from src.opd_loss import forward_kl_loss

# These three-token distributions illustrate the formula. They are synthetic.
student_example = torch.tensor([[[0.5, 0.3, 0.2]]]).log().requires_grad_()
teacher_example = torch.tensor([[[0.7, 0.2, 0.1]]]).log()
example_loss = forward_kl_loss(student_example, teacher_example, teacher_temperature=1.0)
print("Illustrative KL in nats", example_loss.item())
# Shape [1, 1, 3] means batch, response positions, vocabulary entries.
example_loss.backward()
assert torch.isfinite(student_example.grad).all()

## 4. Keep token alignment and endings explicit

Logits at position $t$ predict the next token. The first response token is located
at `prompt_length`, so its predicting logits start at `prompt_length - 1`.

The student uses its question prompt. The teacher uses its pinned non-thinking
chat prefix for that same question. The teacher prefix can be longer. We append
the original response IDs directly to both contexts and use each model's own
causal offset. Both aligned tensors have shape `[1, response_length, vocabulary_size]`.
No prompt position contributes to the response loss.

![Causal alignment](assets/05_qwen38_restart/causal_alignment.png)

The student and teacher can prefer different ending spellings. For this single-turn
protocol, we group their two ending IDs into one stopping event after normalization.

$$
q_{\mathrm{stop}}=\sum_{e\in E}q(e),\qquad p_{\mathrm{stop}}=\sum_{e\in E}p(e).
$$

All ordinary-token contributions remain. We replace the individual ending terms
with $q_{\mathrm{stop}}\log(q_{\mathrm{stop}}/p_{\mathrm{stop}})$ at every response
position. This EOS-aligned loss is an explicit adaptation to GKD. It preserves the
sampled IDs, full output heads, teacher detachment, and response averaging.

![Ending probability illustration](assets/05_qwen38_restart/eos_probabilities.png)

In [ ]:
# Inspect one actual reference boundary before any training models load.
example = state["splits"]["train"][0]
prompt_length = example["prompt_length"]
response_token_ids = example["input_ids"][prompt_length:]
print("Student prompt length", prompt_length)
print("Teacher prompt length", len(example["teacher_prompt_token_ids"]))
print("First response token", repr(state["tokenizer"].decode(response_token_ids[:1])))
print("Student predicting position", prompt_length - 1)
print("Teacher predicting position", len(example["teacher_prompt_token_ids"]) - 1)
assert len(response_token_ids) == example["target_length"]

## 5. Qualify, measure, and pilot

First run 05's real-model GPU qualification. Then run these two pilots from the
same verified SFT checkpoint. Each pilot has its own optimizer, RNG state,
signature, and recovery directory. The teacher stays frozen and is loaded once.

Within an on-policy update, all eight responses are generated before the weights
change. The teacher scores each original response prefix. The student recomputes
its logits with gradients, accumulates the response-averaged loss, clips gradients,
and takes one optimizer step. The full shared implementation is in
[opd_runner.py](../src/opd_runner.py).

Validation occurs every eight updates. A pause is requested when strict accuracy
drops more than five percentage points, reference NLL exceeds 110% of baseline,
or more than eight of the latest 32 sampled responses hit the response limit.
These are engineering safeguards and leave gains to be established by evaluation.

![Checkpoint and validation decisions](assets/05_qwen38_restart/checkpoint_safety.png)

In [ ]:
from src.opd_runner import run_training_arm as train_arm

# The shared loop receives an explicit context containing the checkpoint,
# datasets, evaluation policy, model factory, dtype, and recovery settings.
# run_distillation builds this context and invokes the same function below.
print("Shared implementation", train_arm.__module__ + "." + train_arm.__name__)

In [ ]:
# Pilot mode runs the two predeclared learning rates. A full run needs its saved folder.
# Inspect mode performs no model loading or optimizer updates.
if MODE == "train":
    experiment_result = run_distillation(
        state,
        sft_run_directory=SFT_RUN_DIRECTORY,
        run_directory=RESUME_RUN_DIRECTORY,
        stage=RUN_STAGE,
    )
    print("Run", experiment_result["directory"])
    print("Status", experiment_result["status"])
    print("Selected learning rate", experiment_result.get("selected_learning_rate"))
    # Raw histories are saved to disk. Display compact, measured pilot summaries.
    if experiment_result.get("pilots"):
        display(
            pd.DataFrame(
                [
                    {
                        "learning_rate": rate,
                        "status": result["status"],
                        "updates": result["completed_steps"],
                        "strict_validation_accuracy": result.get("final_validation", {}).get(
                            "strict_accuracy"
                        ),
                        "reference_nll": result.get("final_validation_reference_nll"),
                    }
                    for rate, result in experiment_result["pilots"].items()
                ]
            )
        )
else:
    experiment_result = None
    print("Inspection passed. Real-model qualification and training remain pending.")

## 6. Continue only a qualifying recipe

Among passing 32-update pilots, select the highest strict validation accuracy,
then the lowest NLL, then the lower learning rate. Set `RUN_STAGE = "full"` and
`RESUME_RUN_DIRECTORY` to the printed pilot parent folder. The selected on-policy
arm continues its original 128-update scheduler and RNG. The supervised-KD control
starts from the same SFT checkpoint with the selected recipe.

A quality-paused optimizer stays paused. Its unfinished validation can resume.
A walltime pause can continue from its last complete optimizer boundary. New runs
write pause metadata, per-question evaluation records, progress logs, and small
reports automatically. A ten-minute SLURM reserve protects checkpoint publication.

The supervised control's responses come from references. They have no sampled
response-cap failures; generated validation still measures its stopping behavior.

## 7. Compare the fixed final results

Held-out generation opens only after both fixed full arms complete. Evaluate the
original student, supervised starting student, supervised-KD student, on-policy
student, and teacher under the same greedy policy. Report strict numeric matching,
format compliance, exploratory prose matching, and token-limit stops separately.

Paired bootstrap intervals resample questions. This one-seed, 256-question experiment
can show an observed gain or regression, with uncertainty. Its conclusions apply to
this setup. Save raw responses so every checker decision can be replayed.

If both pilots fail, preserve the validation outcome and stop this predeclared
experiment. The reserved test remains available for a subsequently fixed study.

In [ ]:
# Read only results that actually exist. Every displayed row comes from a completed GPU run.
if experiment_result and experiment_result.get("status") == "completed":
    display(
        pd.DataFrame(
            [
                {
                    "model": name,
                    "strict_accuracy": scores["strict"]["accuracy"],
                    "format_compliance": scores["strict"]["parse_rate"],
                    "prose_numeric_accuracy": scores["prose"]["accuracy"],
                    "token_limit_stops": scores["prose"]["number_token_limit_stops"],
                }
                for name, scores in experiment_result["metrics"].items()
            ]
        )
    )
    print(experiment_result["paired_comparisons"])
else:
    print("No completed final held-out comparison in this kernel.")
# These files are drawn from saved measurements, including paused runs.
if experiment_result:
    from IPython.display import Image

    for figure_name in ("measured_training.png", "measured_final_comparison.png"):
        figure_path = Path(experiment_result["directory"]) / figure_name
        if figure_path.is_file():
            display(Image(filename=str(figure_path)))

## 8. Preserve and interpret the evidence

Reports contain configuration, exact split identities, source fingerprints,
training histories, evaluations, and recorded pauses. Checkpoint folders additionally
contain model and optimizer state. Keep both on persistent storage. The small ZIP
supports review and contains no model weights.

The [previous measured histories](assets/04_on_policy_distillation/recorded_eos_comparison/README.md)
show why a decreasing KL value alone is insufficient. Grouped endings helped the
old models stop, while arithmetic accuracy still declined at update 8.

![Historical pilot measurements](assets/04_on_policy_distillation/pilot_history.png)

*These measurements belong to the original Qwen3 models. New-model measurements
will be recorded under their own profile and experiment signature.*

## What we achieved, in the simplest terms

The student writes an answer. The frozen teacher reads exactly what the student
has written so far and supplies better-informed next-token choices. Only the student
learns from the comparison. A reference-response control helps test whether using
the student's own prefixes makes a difference.

Inspection verifies preparation and alignment. CPU tests exercise the same runner
with tiny random models. The real Amarel run must establish memory use, training
behavior, stopping, and answer quality. Record the actual outcome, including failures,
then update the conclusions and repository evidence.